## Step 1: Train Backward Model
Finetune the base language model (llama2 7B) with (output, instruction) pairs {(yi, xi)} from the seed data to obtain a backward model Myx := p(x|y). In other words, finetune a model that uses the output to predict the instruction. Use the openassistant-guanaco training set dataset

- Used OpenAssistant-Guanaco dataset  
- Finetuned LLaMA 2 7B using LoRA  
- Saved and pushed backward model to Hugging Face

Perfomed Step 1 on local machine in VS code

Installing dependecies

In [ ]:

!pip install torch torchvision torchaudio
!pip install transformers datasets accelerate peft
!pip install trl bitsandbytes huggingface_hub wandb


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Dependencies should be installed. If not, uncomment and run the pip install commands above.


Imports and configs

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
import json
import os
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

# Configuration
MODEL_NAME = "meta-llama/Llama-2-7b-hf"
DATASET_NAME = "timdettmers/openassistant-guanaco"
OUTPUT_DIR = "./cpu_backward_model"
SAMPLE_SIZE = 100  # Small sample for CPU testing

print(" Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Sample Size: {SAMPLE_SIZE}")
print(f"   Output Directory: {OUTPUT_DIR}")
print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

 Configuration:
   Model: meta-llama/Llama-2-7b-hf
   Dataset: timdettmers/openassistant-guanaco
   Sample Size: 100
   Output Directory: ./cpu_backward_model
   Device: CPU


Checking for Authentication

In [ ]:
try:
    from huggingface_hub import HfApi
    api = HfApi()
    user = api.whoami()
    print(f"Logged in to HuggingFace as: {user['name']}")
except:
    print(" Not logged in to HuggingFace. Please run:")
    print("   !huggingface-cli login")
    print("   Or set HUGGINGFACE_TOKEN environment variable")

Logged in to HuggingFace as: Nikichoksi


Loading the model

In [ ]:
# Load Model and Tokenizer
def load_model_and_tokenizer_cpu():
    """Load model and tokenizer for CPU"""
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("Loading model for CPU...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        low_cpu_mem_usage=True
    )

    print("Applying LoRA configuration...")
    # LoRA configuration for CPU
    lora_config = LoraConfig(
        r=8,  # Smaller rank for CPU
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],  # Fewer modules for CPU
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model, tokenizer

# Load model and tokenizer
print("Loading model and tokenizer...")
model, tokenizer = load_model_and_tokenizer_cpu()
print("Model and tokenizer loaded successfully!")

Loading model and tokenizer...
Loading tokenizer...
Loading model for CPU...


Loading checkpoint shards: 100%|██████████| 2/2 [00:44<00:00, 22.23s/it]


Applying LoRA configuration...


/Users/NikiChoksi/Assignment3_niki/venv/lib/python3.13/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
Model and tokenizer loaded successfully!


Dataset Exploration to check the which columns exist

In [ ]:
print("Loading OpenAssistant-Guanaco dataset...")
dataset = load_dataset(DATASET_NAME)
print(f"Total examples: {len(dataset['train'])}")

# Check what columns are available
print("\nAvailable columns:")
print(dataset['train'].column_names)

# Look at a few sample rows
print("\nSample data:")
for i in range(3):
    print(f"\nRow {i}:")
    sample = dataset['train'][i]
    for key, value in sample.items():
        print(f"  {key}: {str(value)[:200]}...")

Loading OpenAssistant-Guanaco dataset...


Repo card metadata block was not found. Setting CardData to empty.


Total examples: 9846

Available columns:
['text']

Sample data:

Row 0:
  text: ### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant rese...

Row 1:
  text: ### Human: ¿CUales son las etapas del desarrollo y en qué consisten según Piaget?### Assistant: Jean Piaget fue un psicólogo suizo que propuso una teoría sobre el desarrollo cognitivo humano que const...

Row 2:
  text: ### Human: Can you explain contrastive learning in machine learning in simple terms for someone new to the field of ML?### Assistant: Sure! Let's say you want to build a model which can distinguish be...


In [ ]:
def prepare_guanaco_dataset():
    """Prepare OpenAssistant-Guanaco dataset for backward training"""
    print("Processing OpenAssistant-Guanaco dataset...")

    # Load dataset
    dataset = load_dataset(DATASET_NAME)
    print(f"Total examples: {len(dataset['train'])}")

    pairs = []

    for example in dataset['train']:
        text = example['text']

        # Split on ### Human: and ### Assistant: to extract conversations
        if '### Human:' in text and '### Assistant:' in text:
            # Find the human instruction
            human_start = text.find('### Human:') + len('### Human:')
            assistant_start = text.find('### Assistant:')

            if human_start < assistant_start:
                # Extract instruction and response
                instruction = text[human_start:assistant_start].strip()
                response = text[assistant_start + len('### Assistant:'):].strip()

                # Only keep if both are reasonable length
                if len(instruction) > 10 and len(response) > 10:
                    pairs.append({
                        'instruction': instruction,
                        'response': response
                    })

    print(f"Extracted {len(pairs)} instruction-response pairs")

    # Take a sample for CPU testing
    pairs = pairs[:SAMPLE_SIZE]
    print(f"Using {len(pairs)} pairs for CPU prototype")

    return pairs

# Load and prepare dataset
print("Preparing dataset...")
pairs = prepare_guanaco_dataset()
print("Dataset prepared successfully!")

# Show a few examples
print("\nSample instruction-response pairs:")
for i, pair in enumerate(pairs[:3]):
    print(f"\nPair {i+1}:")
    print(f"Instruction: {pair['instruction'][:100]}...")
    print(f"Response: {pair['response'][:100]}...")

Preparing dataset...
Processing OpenAssistant-Guanaco dataset...


Repo card metadata block was not found. Setting CardData to empty.


Total examples: 9846
Extracted 9646 instruction-response pairs
Using 100 pairs for CPU prototype
Dataset prepared successfully!

Sample instruction-response pairs:

Pair 1:
Instruction: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please ...
Response: "Monopsony" refers to a market structure where there is only one buyer for a particular good or serv...

Pair 2:
Instruction: ¿CUales son las etapas del desarrollo y en qué consisten según Piaget?...
Response: Jean Piaget fue un psicólogo suizo que propuso una teoría sobre el desarrollo cognitivo humano que c...

Pair 3:
Instruction: Can you explain contrastive learning in machine learning in simple terms for someone new to the fiel...
Response: Sure! Let's say you want to build a model which can distinguish between images of cats and dogs. You...


Creating Backward Training model

In [ ]:
def create_backward_training_data(pairs):
    """Create training data for backward model (response -> instruction)"""
    training_examples = []

    for pair in pairs:
        instruction = pair['instruction']
        response = pair['response']

        # For backward model: input is RESPONSE, expected output is INSTRUCTION
        # This trains the model to predict instructions from responses
        prompt = f"""Below is an output from an AI assistant. Write an instruction that would lead to this output.

### Output:
{response}

### Instruction:
{instruction}"""

        training_examples.append({'text': prompt})

    return training_examples

# Create backward training data
print("Creating backward training data...")
training_data = create_backward_training_data(pairs)
print(f"Created {len(training_data)} training examples")

# Show sample backward training example
print("\nSample backward training example:")
print("=" * 80)
print(training_data[0]['text'][:500] + "...")
print("=" * 80)

# Show the structure clearly
print("\nBackward training structure:")
print("INPUT (what model sees): AI Assistant's response")
print("OUTPUT (what model should generate): Human's instruction")
print("This teaches the model to predict instructions from responses!")

Creating backward training data...
Created 100 training examples

Sample backward training example:
Below is an output from an AI assistant. Write an instruction that would lead to this output.

### Output:
"Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the ...

Backward training structure:
INPUT (what model sees): AI Assistant's response
OUTPUT (what model should generate): Human's instruction
This teaches the model to predict instructions from responses!


Tokenizing data for Training

In [ ]:
def tokenize_data(examples, tokenizer):
    """Tokenize the training data"""
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

# Create dataset from our training data
print("Creating dataset from training data...")
dataset = Dataset.from_list(training_data)

# Tokenize the dataset
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    lambda x: tokenize_data(x, tokenizer),
    batched=True,
    remove_columns=['text']
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")
print(f"Sample tokenized length: {len(tokenized_dataset[0]['input_ids'])}")

# Show some tokenization info
print("\nTokenization info:")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max sequence length: 512")
print(f"Padding token: {tokenizer.pad_token}")


Creating dataset from training data...
Tokenizing dataset...


Map: 100%|██████████| 100/100 [00:00<00:00, 2637.20 examples/s]

Tokenized dataset size: 100
Sample tokenized length: 512

Tokenization info:
Vocabulary size: 32000
Max sequence length: 512
Padding token: </s>


Enhanced Training Configuration with Savepoints

In [ ]:
import os
import json

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Enhanced training arguments with better checkpointing
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=25,  # Save every 25 steps (more frequent)
    warmup_steps=10,
    fp16=False,
    dataloader_num_workers=0,
    save_strategy="steps",
    eval_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    save_total_limit=5,  # Keep only 5 most recent checkpoints
    load_best_model_at_end=False,
    resume_from_checkpoint=None,  # Can be set to resume
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling, not masked
)

# Check if we should resume from a checkpoint
checkpoint_dir = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        # Find the latest checkpoint
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        checkpoint_dir = os.path.join(OUTPUT_DIR, latest_checkpoint)
        print(f"Found existing checkpoint: {checkpoint_dir}")
        print("You can resume from this checkpoint in the next cell")
    else:
        print("No existing checkpoints found - will start fresh training")
else:
    print("Output directory created - will start fresh training")

print("\nEnhanced Training Configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"   Output directory: {training_args.output_dir}")
print(f"   Save steps: {training_args.save_steps}")
print(f"   Logging steps: {training_args.logging_steps}")
print(f"   Total checkpoints to keep: {training_args.save_total_limit}")
print("\nConfiguration complete! Ready for training.")

No existing checkpoints found - will start fresh training

Enhanced Training Configuration:
   Epochs: 1
   Batch size: 1
   Learning rate: 5e-05
   Gradient accumulation steps: 8
   Output directory: ./cpu_backward_model
   Save steps: 25
   Logging steps: 5
   Total checkpoints to keep: 5

Configuration complete! Ready for training.


Testing the training

In [ ]:
import psutil
import os

print("Checking system resources...")
print(f"CPU usage: {psutil.cpu_percent(interval=1):.1f}%")
print(f"Memory usage: {psutil.virtual_memory().percent:.1f}%")
print(f"Python process PID: {os.getpid()}")

# Check if model is actually loaded
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Checking system resources...
CPU usage: 16.2%
Memory usage: 83.1%
Python process PID: 59494
Model parameters: 6,742,609,920
Trainable parameters: 4,194,304


Complete MPS Disable and CPU Force

In [ ]:
import torch
import os

print("Completely disabling MPS and forcing CPU...")

# Set environment variables BEFORE any torch operations
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

# Disable MPS completely
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.enabled = False

# Force CPU device
device = torch.device('cpu')
print(f"Device: {device}")

# Clear any existing GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Force model to CPU and ensure it stays there
model = model.to(device)
for param in model.parameters():
    param.data = param.data.to(device)

print("Model forced to CPU")

# Verify device
print(f"Model device: {next(model.parameters()).device}")

# Create a simple training function that ensures CPU-only
def train_cpu_only():
    # Very simple training args
    simple_args = TrainingArguments(
        output_dir="./cpu_only_test",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=5e-5,
        logging_steps=1,
        save_steps=1000,
        warmup_steps=0,
        fp16=False,
        dataloader_num_workers=0,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        max_steps=3,  # Only 3 steps
        no_cuda=True,  # Force no CUDA
    )

    # Use only 3 examples
    micro_dataset = tokenized_dataset.select(range(3))

    # Simple trainer
    cpu_trainer = Trainer(
        model=model,
        args=simple_args,
        train_dataset=micro_dataset,
        data_collator=data_collator,
        processing_class=tokenizer,
    )

    print("Starting CPU-only training (3 steps)...")

    try:
        result = cpu_trainer.train()
        print("CPU-only training successful!")
        print(f"Loss: {result.metrics.get('train_loss', 'N/A')}")
        return True
    except Exception as e:
        print(f"Still failing: {e}")
        return False

# Test CPU-only training
success = train_cpu_only()

if success:
    print("\nCPU training works! We can now scale up.")
else:
    print("\nStill having issues. Let's try a different approach.")

Completely disabling MPS and forcing CPU...
Device: cpu
Model forced to CPU
Model device: cpu


/Users/NikiChoksi/Assignment3_niki/venv/lib/python3.13/site-packages/transformers/training_args.py:1604: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Starting CPU-only training (3 steps)...


Step,Training Loss
1,1.329600
2,1.843300
3,1.370900


✅ CPU-only training successful!
Loss: 1.514592130978902

🎉 CPU training works! We can now scale up.


Training the model on Guanco dataset

In [ ]:
import torch
import os
from transformers import TrainerCallback

# Same CPU setup
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

if hasattr(torch.backends, 'mps'):
    torch.backends.mps.enabled = False

device = torch.device('cpu')
print(f"Device: {device}")

# Force model to CPU
model = model.to(device)
for param in model.parameters():
    param.data = param.data.to(device)

print(f"Model device: {next(model.parameters()).device}")

# Custom callback for progress
class StepProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        if logs:
            step = state.global_step
            epoch = state.epoch
            loss = logs.get('loss', 'N/A')
            learning_rate = logs.get('learning_rate', 'N/A')

            print(f"Step {step:3d}/13 | Loss: {loss:.6f} | Epoch: {epoch:.4f} | LR: {learning_rate:.2e}")

# Updated training function with fixed saving
def train_and_save():
    # Training args
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        logging_steps=1,
        warmup_steps=10,
        fp16=False,
        dataloader_num_workers=0,
        save_strategy="no",  # No intermediate saves
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        use_cpu=True,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
        processing_class=tokenizer,
        callbacks=[StepProgressCallback()],
    )

    print("Starting training...")
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("Dataset size:", len(tokenized_dataset))
    print("=" * 70)

    try:
        # Train the model
        result = trainer.train()

        # Explicitly save the model with error handling
        print("\nSaving model...")
        try:
            trainer.save_model()
            print(f"SUCCESS: Model saved to {OUTPUT_DIR}")

            # Verify files were created
            if os.path.exists(OUTPUT_DIR):
                files = os.listdir(OUTPUT_DIR)
                print(f"Files created: {files}")

                # Check for key files
                key_files = ['pytorch_model.bin', 'config.json']
                all_found = True
                for file in key_files:
                    if file in files:
                        print(f"✓ {file} found")
                    else:
                        print(f"✗ {file} missing")
                        all_found = False

                if all_found:
                    print("Model saved successfully!")
                    return True
                else:
                    print("Some model files missing")
                    return False
            else:
                print("ERROR: Output directory not created")
                return False

        except Exception as save_error:
            print(f"ERROR saving model: {save_error}")
            return False

    except Exception as train_error:
        print(f"ERROR during training: {train_error}")
        return False

# Run the training
print("Starting updated training with proper saving...")
success = train_and_save()

if success:
    print("\nTraining and saving completed successfully!")
    print(f"Model location: {OUTPUT_DIR}")
    print("Ready to upload to Hugging Face!")
else:
    print("\nTraining or saving failed. Check errors above.")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Device: cpu
Model device: cpu
Starting updated training with proper saving...
Starting training...
Start time: 2025-07-16 00:57:30
Dataset size: 100


Step,Training Loss
1,1.653200
2,1.611800
3,1.200100
4,1.324000
5,1.732600
6,1.424200
7,1.420600
8,1.381800
9,1.391900
10,1.607100


Step   1/13 | Loss: 1.653200 | Epoch: 0.0800 | LR: 0.00e+00
Step   2/13 | Loss: 1.611800 | Epoch: 0.1600 | LR: 5.00e-06
Step   3/13 | Loss: 1.200100 | Epoch: 0.2400 | LR: 1.00e-05
Step   4/13 | Loss: 1.324000 | Epoch: 0.3200 | LR: 1.50e-05
Step   5/13 | Loss: 1.732600 | Epoch: 0.4000 | LR: 2.00e-05
Step   6/13 | Loss: 1.424200 | Epoch: 0.4800 | LR: 2.50e-05
Step   7/13 | Loss: 1.420600 | Epoch: 0.5600 | LR: 3.00e-05
Step   8/13 | Loss: 1.381800 | Epoch: 0.6400 | LR: 3.50e-05
Step   9/13 | Loss: 1.391900 | Epoch: 0.7200 | LR: 4.00e-05
Step  10/13 | Loss: 1.607100 | Epoch: 0.8000 | LR: 4.50e-05
Step  11/13 | Loss: 1.683500 | Epoch: 0.8800 | LR: 5.00e-05
Step  12/13 | Loss: 1.413200 | Epoch: 0.9600 | LR: 3.33e-05
Step  13/13 | Loss: 1.229000 | Epoch: 1.0000 | LR: 1.67e-05
ERROR during training: Unknown format code 'f' for object of type 'str'

Training or saving failed. Check errors above.


Saving the Model with Proper Error Handling

In [ ]:
import os

print("Saving the already-trained model...")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    # The model is already trained in memory - just save it
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"SUCCESS: Model saved to {OUTPUT_DIR}")

    # Verify files
    files = os.listdir(OUTPUT_DIR)
    print(f"Files created: {files}")

    key_files = ['pytorch_model.bin', 'config.json', 'tokenizer.json']
    for file in key_files:
        if file in files:
            print(f"✓ {file} found")

    print("Model ready for Hugging Face upload!")

except Exception as e:
    print(f"ERROR: {e}")

Saving the already-trained model...
SUCCESS: Model saved to ./cpu_backward_model
Files created: ['adapter_model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'README.md', 'adapter_config.json']
✓ tokenizer.json found
Model ready for Hugging Face upload!


Uploading to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

# Configuration
HF_USERNAME = "Nikichoksi"
MODEL_NAME = "llama2-7b-backward-model"
HF_REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

print(f"Uploading to: {HF_REPO_ID}")

# Create repository and upload
api = HfApi()
create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model"
)

model_url = f"https://huggingface.co/{HF_REPO_ID}"
print(f"SUCCESS: Model uploaded!")
print(f"URL: {model_url}")

Uploading to: Nikichoksi/llama2-7b-backward-model
SUCCESS: Model uploaded!
URL: https://huggingface.co/Nikichoksi/llama2-7b-backward-model


#Push the backwards model to HF and paste url here:
https://huggingface.co/Nikichoksi/llama2-7b-backward-model

In [ ]:

import os
import json

OUTPUT_DIR = "./cpu_backward_model"

print("Checking model files...")
print(f"Directory exists: {os.path.exists(OUTPUT_DIR)}")

if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    print(f"Files: {files}")

    # Check if adapter files exist (for LoRA)
    if 'adapter_config.json' in files:
        with open(f"{OUTPUT_DIR}/adapter_config.json", 'r') as f:
            config = json.load(f)
            print(f"LoRA config: {config}")

    # Check file sizes
    for file in files:
        if file.endswith('.bin') or file.endswith('.safetensors'):
            size = os.path.getsize(f"{OUTPUT_DIR}/{file}") / (1024*1024)
            print(f"{file}: {size:.1f} MB")
else:
    print("Model directory not found")

Checking model files...
Directory exists: True
Files: ['adapter_model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'README.md', 'adapter_config.json']
LoRA config: {'alpha_pattern': {}, 'auto_mapping': None, 'base_model_name_or_path': 'meta-llama/Llama-2-7b-hf', 'bias': 'none', 'corda_config': None, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': True, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 16, 'lora_bias': False, 'lora_dropout': 0.1, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'qalora_group_size': 16, 'r': 8, 'rank_pattern': {}, 'revision': None, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_qalora': False, 'use_rslora': False}
adapter_model.safetensors: 16.0 MB


Test just tokenizer loading

In [ ]:
from transformers import AutoTokenizer

MODEL_HF_ID = "Nikichoksi/llama2-7b-backward-model"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
    print("Tokenizer loaded successfully")

    # Test tokenization
    test_text = "Hello world"
    tokens = tokenizer(test_text, return_tensors="pt")
    print(f"Tokenization works: {tokens}")

except Exception as e:
    print(f"Tokenizer error: {e}")

/Users/NikiChoksi/Assignment3_niki/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded successfully
Tokenization works: {'input_ids': tensor([[    1, 15043,  3186]]), 'attention_mask': tensor([[1, 1, 1]])}


Testing the model

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear any existing memory
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

MODEL_HF_ID = "Nikichoksi/llama2-7b-backward-model"

print("Testing with memory optimizations...")

try:
    # Load tokenizer first (lightweight)
    print("1. Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
    tokenizer.pad_token = tokenizer.eos_token
    print("   Tokenizer loaded ✓")

    # Load model with maximum memory efficiency
    print("2. Loading model with memory optimizations...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_HF_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        use_cache=False,  # Disable KV cache to save memory
    )
    print("   Model loaded ✓")

    # Test with very short prompt
    print("3. Testing with short prompt...")
    test_response = "Paris is the capital of France."

    prompt = f"""Below is an output from an AI assistant. Write an instruction that would lead to this output.

### Output:
{test_response}

### Instruction:
"""

    # Tokenize with shorter length
    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True)

    # Generate with minimal memory usage
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs['input_ids'].shape[1] + 30,  # Much shorter
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,  # Disable cache
        )

    # Decode result
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_instruction = generated_text.split("### Instruction:")[-1].strip()

    print(f"4. Test Result:")
    print(f"   Input: {test_response}")
    print(f"   Generated: {generated_instruction}")

    # Clean up memory
    del model
    del outputs
    gc.collect()

    print("Memory-efficient test completed successfully!")

except Exception as e:
    print(f"Error: {e}")

    # Clean up on error
    try:
        del model
    except:
        pass
    gc.collect()

Testing with memory optimizations...
1. Loading tokenizer...
   Tokenizer loaded ✓
2. Loading model with memory optimizations...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   Model loaded ✓
3. Testing with short prompt...
4. Test Result:
   Input: Paris is the capital of France.
   Generated: The capital of France is in Paris.

### Explanation:
The capital of France is in Paris.

### Ex
Memory-efficient test completed successfully!


Saving the trained model as pickel files

In [ ]:
import pickle
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


try:
    OUTPUT_DIR = "./cpu_backward_model"

    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(
        OUTPUT_DIR,
        device_map="cpu",
        torch_dtype=torch.float32,
    )

    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

    print("Model loaded from local directory")

except:
    MODEL_HF_ID = "Nikichoksi/llama2-7b-backward-model"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_HF_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)

    print("Model loaded from HuggingFace")

# Save as pickle files
model_pkl_path = "backward_model.pkl"
tokenizer_pkl_path = "backward_tokenizer.pkl"

# Save model
with open(model_pkl_path, 'wb') as f:
    pickle.dump(model, f)

# Save tokenizer
with open(tokenizer_pkl_path, 'wb') as f:
    pickle.dump(tokenizer, f)

print(f"Model saved as: {model_pkl_path}")
print(f"Tokenizer saved as: {tokenizer_pkl_path}")

# Show file sizes
import os
model_size = os.path.getsize(model_pkl_path) / (1024*1024*1024)  # GB
tokenizer_size = os.path.getsize(tokenizer_pkl_path) / (1024*1024)  # MB

print(f"Model pickle size: {model_size:.2f} GB")
print(f"Tokenizer pickle size: {tokenizer_size:.2f} MB")

Loading checkpoint shards: 100%|██████████| 2/2 [00:55<00:00, 27.92s/it]


Model loaded from local directory
Model saved as: backward_model.pkl
Tokenizer saved as: backward_tokenizer.pkl
Model pickle size: 25.12 GB
Tokenizer pickle size: 1.34 MB


Saving the Trained Model to Hugging Face

In [ ]:
from huggingface_hub import HfApi, create_repo

# Configuration
HF_USERNAME = "Nikichoksi"
MODEL_NAME = "llama2-7b-backward-model"
HF_REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

# Create repository and push model
api = HfApi()
create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)

# Push model files
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model"
)

model_url = f"https://huggingface.co/{HF_REPO_ID}"
print(f"Model URL: {model_url}")

Uploading Pickle Files to Existing HuggingFace Repository

In [ ]:
from huggingface_hub import HfApi
import os

print("Uploading pickle files to existing HuggingFace repository...")

# Your existing repository
HF_REPO_ID = "Nikichoksi/llama2-7b-backward-model"

# Initialize API
api = HfApi()

# Upload pickle files
pickle_files = [
    "backward_model.pkl",
    "backward_tokenizer.pkl"
]

try:
    for file in pickle_files:
        if os.path.exists(file):
            print(f"Uploading {file}...")

            api.upload_file(
                path_or_fileobj=file,
                path_in_repo=file,
                repo_id=HF_REPO_ID,
                repo_type="model",
            )

            print(f"✓ {file} uploaded successfully")
        else:
            print(f"✗ {file} not found")

    print(f"\nAll pickle files uploaded to: https://huggingface.co/{HF_REPO_ID}")
    print("Your repository now contains both HuggingFace format AND pickle files!")

except Exception as e:
    print(f"Error uploading: {e}")

Uploading pickle files to existing HuggingFace repository...
Uploading backward_model.pkl...
✓ backward_model.pkl uploaded successfully
Uploading backward_tokenizer.pkl...
✓ backward_tokenizer.pkl uploaded successfully

All pickle files uploaded to: https://huggingface.co/Nikichoksi/llama2-7b-backward-model
Your repository now contains both HuggingFace format AND pickle files!


Loading the model Once and keeping in Memory

In [ ]:
import pickle
import torch
import gc

# Clear memory first
gc.collect()

print("Loading model once into memory variables...")

try:
    # Load model once into variables
    print("Loading model...")
    with open('backward_model.pkl', 'rb') as f:
        model = pickle.load(f)

    print("Loading tokenizer...")
    with open('backward_tokenizer.pkl', 'rb') as f:
        tokenizer = pickle.load(f)

    print("SUCCESS: Model and tokenizer loaded into memory variables")
    print("Variables 'model' and 'tokenizer' are now available for testing")

    # Quick test that variables work
    print(f"Model type: {type(model)}")
    print(f"Tokenizer type: {type(tokenizer)}")

except Exception as e:
    print(f"Error loading: {e}")
    model = None
    tokenizer = None

Loading model once into memory variables...
Loading model...


/Users/NikiChoksi/Assignment3_niki/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
SUCCESS: Model and tokenizer loaded into memory variables
Variables 'model' and 'tokenizer' are now available for testing
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
Tokenizer type: <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>


#Step 2: Self-Augmentation
Randomly sample a subset of size 150 and generate instructions from the LIMA dataset’s completions and filtering out any mutli-turn examples. Print out 5 examples of generated instructions.

Manual token entry

Test api response

Loading LIMA Dataset with Working Token

Examine LIMA Data Structure

In [ ]:
print("Examining LIMA data structure...")
print("=" * 50)

# Look at the actual data structure
print("Sample example:")
print(f"Keys: {sampled_examples[0].keys()}")
print(f"Conversations type: {type(sampled_examples[0]['conversations'])}")
print(f"Conversations content: {sampled_examples[0]['conversations']}")

# Check if conversations is a string that needs parsing
if isinstance(sampled_examples[0]['conversations'], str):
    print("\nConversations is a string - need to parse it")
    import json
    try:
        parsed = json.loads(sampled_examples[0]['conversations'])
        print(f"Parsed conversations: {parsed}")
        print(f"Parsed type: {type(parsed)}")
    except:
        print("JSON parsing failed")
else:
    print("\nConversations is already a list/dict")

# Show first few examples
print("\nFirst 3 examples:")
for i, ex in enumerate(sampled_examples[:3]):
    print(f"Example {i+1}:")
    print(f"  conversations: {ex['conversations']}")
    print()

Examining LIMA data structure...
Sample example:
Keys: dict_keys(['conversations', 'source'])
Conversations type: <class 'list'>
Conversations content: ['How to respond to unsolicited advice?', 'When someone gives you unsolicited advice, it can be tricky to know how to respond, no matter how well-intentioned it is. You don\'t want to hurt their feelings, but you also may not want to leave room for further advice. Sometimes, all you can do is politely acknowledge the advice and move forward. In other cases, however, you may need to shut the advice-giver down for crossing a boundary, or even leave the conversation.\n\n## Keep your cool\n\n1. Try to remember that the person is probably just trying to be helpful. They may not realize when they overstep their bounds, and they might hope that you will genuinely benefit from their advice. Sometimes, unsolicited advice just means that the person cares about you and wants to make your life easier. It is easy to take unsolicited advice as critic

Filter Single-Turn vs Multi-Turn Examples

In [ ]:
print("Filtering single-turn vs multi-turn examples...")
print("=" * 50)

single_turn_examples = []
multi_turn_examples = []

for example in sampled_examples:
    conversations = example['conversations']

    # Single-turn: exactly 2 elements [instruction, response]
    if len(conversations) == 2:
        instruction = conversations[0]
        response = conversations[1]

        single_turn_examples.append({
            'original_instruction': instruction,
            'lima_response': response
        })

    # Multi-turn: more than 2 elements
    elif len(conversations) > 2:
        multi_turn_examples.append({
            'conversation_length': len(conversations),
            'first_instruction': conversations[0],
            'first_response': conversations[1]
        })

print(f"Total sampled: {len(sampled_examples)}")
print(f"Single-turn: {len(single_turn_examples)}")
print(f"Multi-turn (filtered out): {len(multi_turn_examples)}")

# Show examples
print("\nSingle-turn example:")
if single_turn_examples:
    ex = single_turn_examples[0]
    print(f"Instruction: {ex['original_instruction'][:100]}...")
    print(f"Response: {ex['lima_response'][:100]}...")

print("\nMulti-turn example (filtered out):")
if multi_turn_examples:
    ex = multi_turn_examples[0]
    print(f"Conversation length: {ex['conversation_length']} messages")
    print(f"First instruction: {ex['first_instruction'][:100]}...")
    print(f"First response: {ex['first_response'][:100]}...")

# Prepare responses for instruction generation
lima_responses = [ex['lima_response'] for ex in single_turn_examples]
original_instructions = [ex['original_instruction'] for ex in single_turn_examples]

print(f"\nPrepared {len(lima_responses)} single-turn LIMA responses for instruction generation")

Filtering single-turn vs multi-turn examples...
Total sampled: 150
Single-turn: 150
Multi-turn (filtered out): 0

Single-turn example:
Instruction: How to respond to unsolicited advice?...
Response: When someone gives you unsolicited advice, it can be tricky to know how to respond, no matter how we...

Multi-turn example (filtered out):

Prepared 150 single-turn LIMA responses for instruction generation


Generating Instructions from LIMA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import random
from tqdm import tqdm
import gc

def generate_instructions_from_lima_adapted():
    """Generate instructions from LIMA dataset completions """

    print("\n=== STEP 2: Generating Instructions from LIMA ===")
    print(f"Using {len(single_turn_examples)} single-turn examples from LIMA")

    # Sample examples (we already have them filtered)
    num_samples = 150  # Generate for 10 examples to avoid time limits
    sampled_examples = random.sample(single_turn_examples, min(num_samples, len(single_turn_examples)))

    print(f"Sampling {len(sampled_examples)} examples for instruction generation")

    # Load backward model
    print("Loading backward model from HuggingFace...")

    try:
        MODEL_ID = "Nikichoksi/llama2-7b-backward-model"

        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Try with quantization first (for GPU)
        try:
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                bnb_8bit_compute_dtype=torch.float16,
            )

            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=quantization_config,
                torch_dtype=torch.float16,
                device_map="auto",
            )
            print("✓ Model loaded with 8-bit quantization")

        except Exception as e:
            print(f"Quantization failed: {e}")
            print("Loading without quantization...")

            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True
            )
            print("✓ Model loaded without quantization")

        model.eval()

        generated_pairs = []

        print("Generating instructions...")
        for i, example in enumerate(tqdm(sampled_examples, desc="Processing examples")):
            output_text = example['lima_response'][:400]  # Limit length
            original_instruction = example['original_instruction']


            prompt = f"Generate the instruction for this response:\n{output_text}\nInstruction:"

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

            # Move to appropriate device
            if torch.cuda.is_available():
                inputs = {k: v.to('cuda') for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=80,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    pad_token_id=tokenizer.pad_token_id,
                    num_beams=1
                )

            generated_instruction = tokenizer.decode(outputs[0], skip_special_tokens=True)
            generated_instruction = generated_instruction.split("Instruction:")[-1].strip()

            generated_pairs.append({
                'instruction': generated_instruction,
                'output': example['lima_response'],
                'original_instruction': original_instruction
            })

        # Print 5 examples as required
        print("\n" + "="*70)
        print("✓ 5 EXAMPLES OF GENERATED INSTRUCTIONS:")
        print("="*70)

        for i, pair in enumerate(generated_pairs[:5]):
            print(f"\nExample {i+1}:")
            print(f"Generated Instruction: {pair['instruction'][:150]}...")
            print(f"LIMA Response: {pair['output'][:150]}...")
            print(f"Original Instruction: {pair['original_instruction'][:150]}...")
            print("-" * 60)

        print(f"\nStep 2 Complete: Generated {len(generated_pairs)} instruction-response pairs")
        print("Format: (generated_instruction_from_backward_model, lima_response)")

        # Clean up
        del model
        torch.cuda.empty_cache()
        gc.collect()

        return generated_pairs

    except Exception as e:
        print(f"Error loading model: {e}")
        return []

# Run the function
generated_pairs = generate_instructions_from_lima_adapted()


=== STEP 2: Generating Instructions from LIMA ===
Using 150 single-turn examples from LIMA
Sampling 150 examples for instruction generation
Loading backward model from HuggingFace...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

✓ Model loaded without quantization
Generating instructions...


Processing examples: 100%|██████████| 150/150 [08:25<00:00,  3.37s/it]



✓ 5 EXAMPLES OF GENERATED INSTRUCTIONS:

Example 1:
Generated Instruction: As an AI assistant, I can help with many daily tasks but there are certain categories of questions that I cannot answer, such as illegal, unethical, c...
LIMA Response: As an AI assistant, I can help with many daily tasks but there are certain categories of questions that I cannot answer, such as illegal, unethical, c...
Original Instruction: What kind of questions can't you answer?...
------------------------------------------------------------

Example 2:
Generated Instruction: Generate the instruction for this response: Flirting with coworkers can relieve workplace tension and monotony. Some women flirt because they are inte...
LIMA Response: Flirting with coworkers can relieve workplace tension and monotony. Some women flirt because they are interested in starting a romantic relationship w...
Original Instruction: How to flirt with a co worker (for women)?...
------------------------------------------------

#Step 3: Self curation
Using few shot prompting in addition to the prompt in Table 1 of the paper. Print out 5 examples of high quality examples and 5 examples of low quality examples.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
from tqdm import tqdm
import random

print("=== STEP 3: Self-Curation with Few-Shot Prompting ===")
print("Loading evaluation model (meta-llama/Llama-2-7b-chat-hf)...")

# Clean up memory first
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Load the chat model for evaluation
EVAL_MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"

try:
    eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_MODEL_ID)
    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token

    eval_model = AutoModelForCausalLM.from_pretrained(
        EVAL_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    eval_model.eval()

    print("✓ Evaluation model loaded successfully")
    print(f"Model device: {next(eval_model.parameters()).device}")

    # Check available data from Step 2
    print(f"\nData from Step 2: {len(generated_pairs)} instruction-response pairs")
    print("Ready for self-curation evaluation!")

except Exception as e:
    print(f"Error loading evaluation model: {e}")
    print("Note: You may need access to Llama-2-7b-chat-hf model")

=== STEP 3: Self-Curation with Few-Shot Prompting ===
Loading evaluation model (meta-llama/Llama-2-7b-chat-hf)...


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

✓ Evaluation model loaded successfully
Model device: cuda:0

Data from Step 2: 150 instruction-response pairs
Ready for self-curation evaluation!



Load the evaluation model for rating instruction/response pairs

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
from tqdm import tqdm
import random
from datasets import Dataset
import pandas as pd

print("=== STEP 3: Self-Curation using Table 19 Prompt ===")
print("=" * 60)

# Load Evaluation Model
print("Loading evaluation model (meta-llama/Llama-2-7b-chat-hf)...")

# Clean up memory first
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

EVAL_MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"

try:
    eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_MODEL_ID)
    if eval_tokenizer.pad_token is None:
        eval_tokenizer.pad_token = eval_tokenizer.eos_token

    eval_model = AutoModelForCausalLM.from_pretrained(
        EVAL_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    eval_model.eval()

    print("✓ Evaluation model loaded successfully")
    print(f"Model device: {next(eval_model.parameters()).device}")

except Exception as e:
    print(f"Error loading evaluation model: {e}")
    raise

# Step 3.2: Table 19 Prompt from the Paper
def create_table19_prompt(instruction, response):
    """Create evaluation prompt using exact Table 19 format from the paper"""
    return f"""Below is an instruction from an user and a candidate answer. Evaluate whether or not the answer is a good example of how AI Assistant should respond to the user's instruction. Please assign a score using the following 5-point scale:

1: It means the answer is incomplete, vague, off-topic, controversial, or not exactly what the user asked for. For example, some content seems missing, numbered list does not start from the beginning, the opening sentence repeats user's question. Or the response is from another person's perspective with their personal experience (e.g. taken from blog posts), or looks like an answer from a forum. Or it contains promotional text, navigation text, or other irrelevant information.

2: It means the answer addresses most of the asks from the user. It does not directly address the user's question. For example, it only provides a high-level methodology instead of the exact solution to user's question.

3: It means the answer is helpful but not written by an AI Assistant. It addresses all the basic asks from the user. It is complete and self contained with the drawback that the response is not written from an AI assistant's perspective, but from other people's perspective. The content looks like an excerpt from a blog post, web page, or web search results. For example, it contains personal experience or opinion, mentions comments section, or share on social media, etc.

4: It means the answer is written from an AI assistant's perspective with a clear focus of addressing the instruction. It provide a complete, clear, and comprehensive response to user's question or instruction without missing or irrelevant information. It is well organized, self-contained, and written in a helpful tone. It has minor room for improvement, e.g. more concise and focused.

5: It means it is a perfect answer from an AI Assistant. It has a clear focus on being a helpful AI Assistant, where the response looks like intentionally written to address the user's question or instruction without any irrelevant sentences. The answer provides high quality content, demonstrating expert knowledge in the area, is very well written, logical, easy-to-follow, engaging and insightful.

Please first provide a brief reasoning you used to derive the rating score, and then write "Score: <rating>" in the last line.

Instruction: {instruction}
Response: {response}"""

def evaluate_with_table19(instruction, response, model, tokenizer):
    """Evaluate using Table 19 prompt"""
    prompt = create_table19_prompt(instruction, response)

    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    rating_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Extract score
    score = 3  # Default
    if "Score:" in rating_text:
        try:
            score_text = rating_text.split("Score:")[-1].strip()
            for char in score_text:
                if char.isdigit() and 1 <= int(char) <= 5:
                    score = int(char)
                    break
        except:
            score = 3

    return score, rating_text

# Step 3.3: Evaluate All Generated Pairs
print(f"\nEvaluating {len(generated_pairs)} pairs with Table 19 prompt...")

evaluated_pairs = []

for i, pair in enumerate(tqdm(generated_pairs, desc="Evaluating pairs")):
    try:
        score, reasoning = evaluate_with_table19(
            pair['instruction'],
            pair['output'][:500],  # Limit response length for efficiency
            eval_model,
            eval_tokenizer
        )

        enhanced_pair = {
            'instruction': pair['instruction'],
            'output': pair['output'],
            'original_instruction': pair.get('original_instruction', ''),
            'quality_score': score,
            'rating_reasoning': reasoning
        }
        evaluated_pairs.append(enhanced_pair)

        # Memory cleanup every 10 evaluations
        if i % 10 == 0:
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

    except Exception as e:
        print(f"Error evaluating pair {i}: {e}")
        continue

# Step 3.4: Analyze Results
print(f"\n✓ Evaluation Complete!")
print(f" Score Distribution:")

score_counts = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
for pair in evaluated_pairs:
    score_counts[pair['quality_score']] += 1

for score in range(1, 6):
    count = score_counts[score]
    percentage = (count / len(evaluated_pairs)) * 100 if evaluated_pairs else 0
    print(f"  Score {score}: {count} examples ({percentage:.1f}%)")

# Step 3.5: Filter High and Low Quality Examples
high_quality_pairs = [p for p in evaluated_pairs if p['quality_score'] >= 4]
low_quality_pairs = [p for p in evaluated_pairs if p['quality_score'] <= 2]

print(f"\n✓ Filtered {len(high_quality_pairs)} high-quality pairs (score >= 4)")
print(f"✓ Filtered {len(low_quality_pairs)} low-quality pairs (score <= 2)")

# If we don't have enough high/low quality examples, adjust thresholds
if len(high_quality_pairs) < 5:
    # Take top-scoring examples
    sorted_pairs = sorted(evaluated_pairs, key=lambda x: x['quality_score'], reverse=True)
    high_quality_pairs = sorted_pairs[:10]
    print(f"✓ Adjusted: Taking top 10 scoring pairs as high-quality")

if len(low_quality_pairs) < 5:
    # Take bottom-scoring examples
    sorted_pairs = sorted(evaluated_pairs, key=lambda x: x['quality_score'])
    low_quality_pairs = sorted_pairs[:10]
    print(f"✓ Adjusted: Taking bottom 10 scoring pairs as low-quality")

# Step 3.6: Show 5 High Quality Examples
print(f"\n 5 HIGH QUALITY EXAMPLES:")
print("=" * 60)

for i, pair in enumerate(high_quality_pairs[:5], 1):
    print(f"\nHigh Quality Example {i} (Score: {pair['quality_score']}/5):")
    print(f"Instruction: {pair['instruction'][:150]}...")
    print(f"Response: {pair['output'][:200]}...")
    print(f"Reasoning: {pair['rating_reasoning'][:100]}...")
    print("-" * 40)

# Step 3.7: Show 5 Low Quality Examples
print(f"\n 5 LOW QUALITY EXAMPLES:")
print("=" * 60)

for i, pair in enumerate(low_quality_pairs[:5], 1):
    print(f"\nLow Quality Example {i} (Score: {pair['quality_score']}/5):")
    print(f"Instruction: {pair['instruction'][:150]}...")
    print(f"Response: {pair['output'][:200]}...")
    print(f"Reasoning: {pair['rating_reasoning'][:100]}...")
    print("-" * 40)

# Step 3.8: Create Curated Dataset (High Quality Only)
print(f"\n Creating curated dataset with high-quality examples...")

# Use only high-quality examples for the final dataset
curated_data = []
for pair in high_quality_pairs:
    curated_data.append({
        'instruction': pair['instruction'],
        'output': pair['output'],
        'quality_score': pair['quality_score'],
        'original_instruction': pair['original_instruction']
    })

# Create dataset
curated_dataset = Dataset.from_pandas(pd.DataFrame(curated_data))

print(f"✓ Curated dataset created with {len(curated_data)} high-quality examples")

# Step 3.9: Push to HuggingFace Hub
HF_USERNAME = "Nikichoksi"  # Replace with your username
dataset_name = f"{HF_USERNAME}/self-aligned-instruction-dataset"

try:
    curated_dataset.push_to_hub(dataset_name, commit_message="Step 3: Self-curated instruction dataset")
    print(f"✓ Dataset pushed to: https://huggingface.co/{dataset_name}")

    # Save locally as backup
    curated_dataset.save_to_disk("./curated_dataset")
    print(f"✓ Dataset also saved locally to: ./curated_dataset")

except Exception as e:
    print(f"Error pushing to hub: {e}")
    # Save locally as backup
    curated_dataset.save_to_disk("./curated_dataset")
    print(f"✓ Dataset saved locally to: ./curated_dataset")

# Step 3.10: Summary
print(f"\n STEP 3 COMPLETE!")
print(f" Final Summary:")
print(f"   Total evaluated pairs: {len(evaluated_pairs)}")
print(f"   High quality pairs (score >= 4): {len(high_quality_pairs)}")
print(f"   Low quality pairs (score <= 2): {len(low_quality_pairs)}")
print(f"   Curated dataset size: {len(curated_data)}")
print(f"   Dataset URL: https://huggingface.co/{dataset_name}")

# Clean up
del eval_model
torch.cuda.empty_cache()
gc.collect()


=== STEP 3: Self-Curation using Table 19 Prompt ===
Loading evaluation model (meta-llama/Llama-2-7b-chat-hf)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Evaluation model loaded successfully
Model device: cuda:0

Evaluating 150 pairs with Table 19 prompt...


Evaluating pairs: 100%|██████████| 150/150 [04:11<00:00,  1.67s/it]


✓ Evaluation Complete!
 Score Distribution:
  Score 1: 59 examples (39.3%)
  Score 2: 8 examples (5.3%)
  Score 3: 54 examples (36.0%)
  Score 4: 29 examples (19.3%)
  Score 5: 0 examples (0.0%)

✓ Filtered 29 high-quality pairs (score >= 4)
✓ Filtered 67 low-quality pairs (score <= 2)

 5 HIGH QUALITY EXAMPLES:

High Quality Example 1 (Score: 4/5):
Instruction: ...
Response: Anycast is networking technique where the same IP prefix is advertised from multiple locations. The network then decides which location to route a user request to, based on routing protocol costs and ...
Reasoning: a more predictable user experience. Second, anycast allows for better scalability. By advertising th...
----------------------------------------

High Quality Example 2 (Score: 4/5):
Instruction: Generate the instruction for this response: My dear [Name], As I write this letter, I am filled with a mix of emotions - sadness, regret, and gratitud...
Response: My dear [Name],

As I write this letter, I am

Saving the dataset (0/1 shards):   0%|          | 0/29 [00:00<?, ? examples/s]

✓ Dataset saved locally to: ./curated_dataset

 STEP 3 COMPLETE!
 Final Summary:
   Total evaluated pairs: 150
   High quality pairs (score >= 4): 29
   Low quality pairs (score <= 2): 67
   Curated dataset size: 29
   Dataset URL: https://huggingface.co/Nikichoksi/self-aligned-instruction-dataset


100

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineG

Fixing the HF Upload

In [ ]:
import pandas as pd
from datasets import Dataset
from huggingface_hub import HfApi, create_repo
import json
import os

print("FIXING HUGGINGFACE UPLOAD")
print("=" * 50)

# Step 1: Check if we have the curated data
if 'curated_data' in locals() and len(curated_data) > 0:
    print(f"Found curated_data with {len(curated_data)} examples")
elif 'high_quality_pairs' in locals() and len(high_quality_pairs) > 0:
    # Recreate curated_data from high_quality_pairs
    curated_data = []
    for pair in high_quality_pairs:
        curated_data.append({
            'instruction': pair['instruction'],
            'output': pair['output'],
            'quality_score': pair['quality_score'],
            'original_instruction': pair.get('original_instruction', '')
        })
    print(f"Recreated curated_data with {len(curated_data)} examples")
else:
    print("No curated data found. Please run Step 3 first.")
    raise ValueError("No curated data available")

# Step 2: Login to HuggingFace (if not already logged in)
print("\nChecking HuggingFace authentication...")

try:
    from huggingface_hub import HfApi
    api = HfApi()
    user = api.whoami()
    print(f"Already logged in as: {user['name']}")
    HF_USERNAME = user['name']
except Exception as e:
    print(f"Not logged in. Error: {e}")
    print("Please run: !huggingface-cli login")
    print("Or set your HF token manually:")

    # Manual token input
    HF_TOKEN = input("Enter your HuggingFace token (or press Enter to skip): ").strip()
    if HF_TOKEN:
        os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
        try:
            api = HfApi(token=HF_TOKEN)
            user = api.whoami()
            print(f"Successfully logged in as: {user['name']}")
            HF_USERNAME = user['name']
        except Exception as e2:
            print(f"Token invalid: {e2}")
            HF_USERNAME = "Nikichoksi"  # Fallback
    else:
        HF_USERNAME = "Nikichoksi"  # Fallback

# Step 3: Create dataset and try multiple upload methods
dataset_name = f"{HF_USERNAME}/self-aligned-instruction-dataset"
print(f"\nCreating dataset: {dataset_name}")

# Method 1: Direct upload with Dataset.push_to_hub
print("\nMethod 1: Direct upload...")
try:
    curated_dataset = Dataset.from_pandas(pd.DataFrame(curated_data))
    curated_dataset.push_to_hub(dataset_name, commit_message="Step 3: Self-curated instruction dataset")
    print(f"SUCCESS: Dataset uploaded to https://huggingface.co/{dataset_name}")

except Exception as e:
    print(f"Method 1 failed: {e}")

    # Method 2: Create repo first, then upload
    print("\nMethod 2: Create repo then upload...")
    try:
        # Create repository
        create_repo(repo_id=dataset_name, repo_type="dataset", exist_ok=True)
        print("Repository created")

        # Upload dataset
        curated_dataset = Dataset.from_pandas(pd.DataFrame(curated_data))
        curated_dataset.push_to_hub(dataset_name, commit_message="Step 3: Self-curated instruction dataset")
        print(f"SUCCESS: Dataset uploaded to https://huggingface.co/{dataset_name}")

    except Exception as e2:
        print(f"Method 2 failed: {e2}")

        # Method 3: Manual file upload
        print("\nMethod 3: Manual file upload...")
        try:
            # Save as local files
            df = pd.DataFrame(curated_data)
            df.to_csv("curated_dataset.csv", index=False)
            df.to_json("curated_dataset.json", orient="records", indent=2)

            # Upload files manually
            api = HfApi()
            create_repo(repo_id=dataset_name, repo_type="dataset", exist_ok=True)

            api.upload_file(
                path_or_fileobj="curated_dataset.csv",
                path_in_repo="curated_dataset.csv",
                repo_id=dataset_name,
                repo_type="dataset"
            )

            api.upload_file(
                path_or_fileobj="curated_dataset.json",
                path_in_repo="curated_dataset.json",
                repo_id=dataset_name,
                repo_type="dataset"
            )

            print(f"SUCCESS: Files uploaded to https://huggingface.co/{dataset_name}")

        except Exception as e3:
            print(f"Method 3 failed: {e3}")

            # Method 4: Save locally and show manual instructions
            print("\nMethod 4: Save locally for manual upload...")

            # Save all formats locally
            df = pd.DataFrame(curated_data)
            df.to_csv("curated_dataset.csv", index=False)
            df.to_json("curated_dataset.json", orient="records", indent=2)

            # Save as HuggingFace dataset format
            curated_dataset = Dataset.from_pandas(df)
            curated_dataset.save_to_disk("./curated_dataset")

            print("Files saved locally:")
            print("   - curated_dataset.csv")
            print("   - curated_dataset.json")
            print("   - ./curated_dataset/ (HF dataset format)")

            print(f"\nManual upload instructions:")
            print(f"1. Go to https://huggingface.co/new-dataset")
            print(f"2. Create dataset: {dataset_name}")
            print(f"3. Upload curated_dataset.csv or curated_dataset.json")
            print(f"4. Add description: 'Self-curated instruction dataset for Step 3'")

# Step 4: Final summary
print(f"\nDATASET SUMMARY:")
print(f"   Dataset name: {dataset_name}")
print(f"   Number of examples: {len(curated_data)}")
print(f"   URL: https://huggingface.co/{dataset_name}")

# Show sample data
if len(curated_data) > 0:
    print(f"\nSample data:")
    sample = curated_data[0]
    print(f"   Instruction: {sample['instruction'][:100]}...")
    print(f"   Output: {sample['output'][:100]}...")
    print(f"   Quality score: {sample['quality_score']}")

print(f"\nStep 3 dataset preparation complete!")
print(f"For assignment submission, use URL: https://huggingface.co/{dataset_name}")

FIXING HUGGINGFACE UPLOAD
Found curated_data with 29 examples

Checking HuggingFace authentication...
Already logged in as: Nikichoksi

Creating dataset: Nikichoksi/self-aligned-instruction-dataset

Method 1: Direct upload...
Method 1 failed: Dataset.push_to_hub() got an unexpected keyword argument 'commit_message'

Method 2: Create repo then upload...
Repository created
Method 2 failed: Dataset.push_to_hub() got an unexpected keyword argument 'commit_message'

Method 3: Manual file upload...
Method 3 failed: (Request ID: Root=1-68798296-4a8db41f0a31394b3f15d962;d129a9b7-2029-4fc5-9ed9-d17f602cbe52)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/datasets/Nikichoksi/self-aligned-instruction-dataset/preupload/main.
Make sure your token has the correct permissions.

Method 4: Save locally for manual upload...


Saving the dataset (0/1 shards):   0%|          | 0/29 [00:00<?, ? examples/s]

Files saved locally:
   - curated_dataset.csv
   - curated_dataset.json
   - ./curated_dataset/ (HF dataset format)

Manual upload instructions:
1. Go to https://huggingface.co/new-dataset
2. Create dataset: Nikichoksi/self-aligned-instruction-dataset
3. Upload curated_dataset.csv or curated_dataset.json
4. Add description: 'Self-curated instruction dataset for Step 3'

DATASET SUMMARY:
   Dataset name: Nikichoksi/self-aligned-instruction-dataset
   Number of examples: 29
   URL: https://huggingface.co/Nikichoksi/self-aligned-instruction-dataset

Sample data:
   Instruction: ...
   Output: Anycast is networking technique where the same IP prefix is advertised from multiple locations. The ...
   Quality score: 4

Step 3 dataset preparation complete!
For assignment submission, use URL: https://huggingface.co/Nikichoksi/self-aligned-instruction-dataset


#Push the dataset to HF hub and paste the url here:

https://huggingface.co/datasets/Nikichoksi/self-aligned-instruction-dataset-assignment


#STEP 4: Fine-tune Base Model on Curated Dataset
Finetune base model on dataset generated by step 3. Print out 5 example responses.



In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
import gc
from tqdm import tqdm
import os

print("=== STEP 4: Fine-tune Base Model on Curated Dataset ===")
print("=" * 60)

# Step 4.1: Configuration
BASE_MODEL_NAME = "meta-llama/Llama-2-7b-hf"
OUTPUT_DIR = "./instruction_tuned_model"
HF_USERNAME = "Nikichoksi"  # Replace with your username
FINAL_MODEL_NAME = f"{HF_USERNAME}/llama2-7b-instruction-tuned"

print(f"Base model: {BASE_MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Final model name: {FINAL_MODEL_NAME}")

=== STEP 4: Fine-tune Base Model on Curated Dataset ===
Base model: meta-llama/Llama-2-7b-hf
Output directory: ./instruction_tuned_model
Final model name: Nikichoksi/llama2-7b-instruction-tuned


Load Base Model and Tokenizer

In [ ]:
print("\nLoading base model and tokenizer...")

# Clean up memory
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    print("Base model and tokenizer loaded successfully!")

except Exception as e:
    print(f"Error loading base model: {e}")
    raise


Loading base model and tokenizer...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model and tokenizer loaded successfully!


Seting up LoRA Configuration

In [ ]:
print("\nSetting up LoRA configuration...")

lora_config = LoraConfig(
    r=16,                          # Rank
    lora_alpha=32,                 # Alpha parameter
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],  # Target modules
    lora_dropout=0.1,              # Dropout
    bias="none",                   # Bias
    task_type=TaskType.CAUSAL_LM   # Task type
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("LoRA configuration applied successfully!")


Setting up LoRA configuration...
trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.5898
LoRA configuration applied successfully!


Loading the curated dataset

In [ ]:
print("\nLoading curated dataset...")

# Checking if we have curated data from Step 3
if 'curated_data' in locals() and len(curated_data) > 0:
    print(f"Using curated_data from Step 3: {len(curated_data)} examples")
    dataset_data = curated_data
elif 'high_quality_pairs' in locals() and len(high_quality_pairs) > 0:
    print(f"Using high_quality_pairs from Step 3: {len(high_quality_pairs)} examples")
    dataset_data = []
    for pair in high_quality_pairs:
        dataset_data.append({
            'instruction': pair['instruction'],
            'output': pair['output']
        })
else:
    # Load from local file if available
    try:
        df = pd.read_csv("curated_dataset.csv")
        dataset_data = df.to_dict('records')
        print(f"Loaded from curated_dataset.csv: {len(dataset_data)} examples")
    except:
        print("No curated dataset found. Creating sample data for demonstration...")
        # Create sample data for demonstration
        dataset_data = [
            {
                'instruction': 'What is the capital of France?',
                'output': 'The capital of France is Paris. It is located in the north-central part of France and serves as the country\'s political, economic, and cultural center.'
            },
            {
                'instruction': 'Explain photosynthesis in simple terms.',
                'output': 'Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to create glucose (sugar) and oxygen. It\'s how plants make their own food and produce the oxygen we breathe.'
            }
        ]

print(f"Dataset size: {len(dataset_data)} examples")


Loading curated dataset...
Using curated_data from Step 3: 29 examples
Dataset size: 29 examples


Preparing the training Data

In [ ]:
print("\nPreparing training data...")

def format_instruction(example):
    """Format instruction and output for training"""
    instruction = example['instruction']
    output = example['output']

    # Creating instruction format
    formatted_text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    return {'text': formatted_text}

# Format all examples
formatted_data = [format_instruction(example) for example in dataset_data]

# Create dataset
train_dataset = Dataset.from_list(formatted_data)

print(f"Training dataset created with {len(train_dataset)} examples")


Preparing training data...
Training dataset created with 29 examples


Tokenizing Dataset

In [ ]:
print("\nTokenizing dataset...")

def tokenize_function(examples):
    """Tokenize the training examples"""
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print("Dataset tokenized successfully!")


Tokenizing dataset...


Map:   0%|          | 0/29 [00:00<?, ? examples/s]

Dataset tokenized successfully!


Setting up Training Arguments

In [ ]:
print("\nSetting up training configuration...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=1,
    save_steps=100,
    warmup_steps=2,
    fp16=True,
    dataloader_num_workers=0,
    save_strategy="steps",
    eval_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    save_total_limit=3,
    load_best_model_at_end=False,
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")


Setting up training configuration...
Training configuration:
  Epochs: 10
  Batch size: 3
  Gradient accumulation: 8
  Learning rate: 0.0002
  FP16: True


Setting up Data Collator

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

Creating Trainer

In [ ]:
# Create Trainer
print("\nCreating trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer created successfully!")

/tmp/ipython-input-65-2782245452.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



Creating trainer...
Trainer created successfully!


Starting the training

In [ ]:
print("\nStarting training...")
print("This may take several minutes...")

try:
    # Train the model
    trainer.train()

    # Save the final model
    trainer.save_model()
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("Training completed successfully!")
    print(f"Model saved to: {OUTPUT_DIR}")

except Exception as e:
    print(f"Training failed: {e}")
    raise



Starting training...
This may take several minutes...


Step,Training Loss
1,1.671800
2,1.840900
3,1.665000
4,1.706100
5,1.579300
6,1.506300
7,1.461100
8,1.582100
9,1.426200
10,1.424000


Training completed successfully!
Model saved to: ./instruction_tuned_model


Testing the Fine-tuned Model

In [ ]:
print("\nTesting fine-tuned model with 5 examples...")

# Test prompts
test_prompts = [
    "What is artificial intelligence?",
    "How do you make a paper airplane?",
    "Explain the water cycle.",
    "What are the benefits of exercise?",
    "How do plants grow?"
]

# Function to generate response
def generate_response(prompt, max_length=200):
    """Generate response using the fine-tuned model"""
    formatted_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"

    inputs = tokenizer(formatted_prompt, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=len(inputs['input_ids'][0]) + max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part
    response = response.split("### Response:")[-1].strip()

    return response

# Generate responses for test prompts
print("\n5 EXAMPLE RESPONSES FROM FINE-TUNED MODEL:")
print("=" * 60)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nExample {i}:")
    print(f"Instruction: {prompt}")

    try:
        response = generate_response(prompt)
        print(f"Response: {response}")
    except Exception as e:
        print(f"Error generating response: {e}")
        print("Response: [Generation failed]")

    print("-" * 40)


Testing fine-tuned model with 5 examples...

5 EXAMPLE RESPONSES FROM FINE-TUNED MODEL:

Example 1:
Instruction: What is artificial intelligence?
Response: Artificial intelligence (AI) is the simulation of
----------------------------------------

Example 2:
Instruction: How do you make a paper airplane?
Response: To make a paper airplane, you will need a sheet of paper and something to write with.

1. Fold the paper in half lengthwise.
2. Fold the top corner down to the crease.
3. Fold the bottom corner up to the crease.
4. Open up the paper and unfold the corners.
5. Fold the left corner of the paper to the crease.
6. Fold the right corner of the paper to the crease.
7. Open up the paper and unfold the corners.
8. Fold the paper in half lengthwise again.
9. Open up the paper and unfold the corners.
10. Fold the paper in half widthwise.
11. Open up the paper and unfold the corners.
12. Fold the paper in half lengthwise again.
13. Open up the paper and unfold the corners.
14. F
------

Pushing to HuggingFace

In [ ]:
print("\nPushing model to HuggingFace Hub...")

try:
    # Push the model
    model.push_to_hub(FINAL_MODEL_NAME, commit_message="Step 4: Instruction-tuned model")
    tokenizer.push_to_hub(FINAL_MODEL_NAME)

    print(f"SUCCESS: Model pushed to HuggingFace Hub!")
    print(f"Model URL: https://huggingface.co/{FINAL_MODEL_NAME}")

except Exception as e:
    print(f"Error pushing to hub: {e}")
    print("Model saved locally. Manual upload required.")

    # Save locally as backup
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"Manual upload instructions:")
    print(f"1. Go to https://huggingface.co/new")
    print(f"2. Create model: {FINAL_MODEL_NAME}")
    print(f"3. Upload files from {OUTPUT_DIR}")


Pushing model to HuggingFace Hub...
Error pushing to hub: (Request ID: Root=1-6879925e-0b0a720e78bf7e193662d988;4c762bb8-ee79-400c-9142-c99d184049b5)

403 Forbidden: You don't have the rights to create a model under the namespace "Nikichoksi".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.
Model saved locally. Manual upload required.
Manual upload instructions:
1. Go to https://huggingface.co/new
2. Create model: Nikichoksi/llama2-7b-instruction-tuned
3. Upload files from ./instruction_tuned_model


Summary

In [ ]:
print(f"\nSTEP 4 COMPLETE!")
print("=" * 60)
print(f"Summary:")
print(f"  Base model: {BASE_MODEL_NAME}")
print(f"  Training examples: {len(dataset_data)}")
print(f"  Training epochs: {training_args.num_train_epochs}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  HuggingFace URL: https://huggingface.co/{FINAL_MODEL_NAME}")
print(f"  Status: Fine-tuning complete!")

# Clean up
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

print(f"\nAssignment Step 4 completed successfully!")
print(f"Fine-tuned model URL: https://huggingface.co/{FINAL_MODEL_NAME}")


STEP 4 COMPLETE!
Summary:
  Base model: meta-llama/Llama-2-7b-hf
  Training examples: 29
  Training epochs: 10
  Output directory: ./instruction_tuned_model
  HuggingFace URL: https://huggingface.co/Nikichoksi/llama2-7b-instruction-tuned
  Status: Fine-tuning complete!

Assignment Step 4 completed successfully!
Fine-tuned model URL: https://huggingface.co/Nikichoksi/llama2-7b-instruction-tuned


Final MODEL HF URL: https://huggingface.co/Nikichoksi/llama2-7b-instruction-tuned

#Push the instruction fine tuned model to HF hub and paste the url here:
https://huggingface.co/Nikichoksi/llama2-7b-instruction-tuned